In [14]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import (
    Compose, Resize, ToTensor, Normalize, ConvertImageDtype, RandomRotation, RandomHorizontalFlip
)
from pathlib import Path
import sys
import pandas as pd
import numpy as np

from tqdm import tqdm

from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
def get_trainer():
    import sys
    project_root = Path.cwd().resolve().parent
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    from base import TorchTrainer
    return TorchTrainer

In [2]:
class VGG19(nn.Module):
    def __init__(self, num_classes=1000):
        super(VGG19, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


print("Total parameters:", sum(p.numel() for p in VGG19().parameters()))

Total parameters: 143667240


In [10]:
from torchviz import make_dot

model = VGG19()

img = torch.randn(1, 3, 224, 224)
output = model(img)

make_dot(output, params=dict(model.named_parameters())).save("model_VGG19.dot")

'model_VGG19.dot'

In [5]:
DATA_DIR = Path('../data/cifar-10')
train_labels = pd.read_csv(DATA_DIR / 'trainLabels.csv')
test_labels = pd.read_csv(DATA_DIR / 'sampleSubmission.csv')
train_folder = DATA_DIR / 'train'
test_folder = DATA_DIR / 'test'

class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]


def get_image_path(labels, folder):
    path_labels = []
    for index, row in labels.iterrows():
        image_id = row['id']
        image_label = row['label']
        image_path = folder / f"{image_id}.png"
        path_labels.append((image_path, image_label))
    return path_labels


train_image_paths = get_image_path(train_labels, train_folder)
test_image_paths = get_image_path(test_labels, test_folder)

ratio = 0.9
train_df = pd.DataFrame(
    train_image_paths[:int(len(train_image_paths) * ratio)],
    columns=['image_path', 'label'])
val_df = pd.DataFrame(
    train_image_paths[int(len(train_image_paths) * ratio):],
    columns=['image_path', 'label'])
test_df = pd.DataFrame(test_image_paths, columns=['image_path', 'label'])

In [16]:
class Cifar10Dataset(Dataset):
    df_map = {
        'train': train_df,
        'val': val_df,
        'test': test_df
    }

    label_to_idx = {label: idx for idx, label in enumerate(class_names)}
    idx_to_label = {idx: label for idx, label in enumerate(class_names)}

    def __init__(self, model="train", transform=None):
        self.data = self.df_map[model]
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path, label = self.data.iloc[idx]
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = self.label_to_idx[label]
        return image, label


IMG_SIZE = 244


def compute_mean_std(dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    mean = 0.0
    std = 0.0
    total_images = 0

    for images, _ in tqdm(loader, desc="Computing mean/std"):
        # batch size (the last batch can have smaller size)
        batch_samples = images.size(0)
        # reshape to (batch_size, channels, width*height)
        images = images.view(batch_samples, images.size(1), -1)
        mean += images.mean(2).sum(0)  # sum of means for each channel
        std += images.std(2).sum(0)  # sum of stds for each channel
        total_images += batch_samples

    mean /= total_images
    std /= total_images

    return mean.tolist(), std.tolist()


mean, std = compute_mean_std(
    Cifar10Dataset(model='train', transform=Compose([
        Resize(IMG_SIZE),
        ToTensor(),
    ])))


# 图像增强可以增加训练数据的多样性，帮助模型更好地泛化，从而提高模型的性能和鲁棒性。
# 1. 随机旋转图像：通过随机旋转图像，可以模拟不同的拍摄角度和姿态，使模型更好地适应各种视角。
# 2. 随机水平翻转图像：通过随机水平翻转图像，可以模拟左右对称的物体，使模型更好地适应不同的方向和姿态。
# 3. 随机裁剪图像：通过随机裁剪图像，可以模拟不同的物体位置和大小，使模型更好地适应各种场景和物体尺寸。
# 4. 随机调整图像亮度、对比度和饱和度：通过随机调整图像的亮度、对比度和饱和度，可以模拟不同的光照条件和环境，使模型更好地适应各种光照变化。
# 5. 随机添加噪声：通过随机添加噪声，可以模拟不同的图像质量和干扰，使模型更好地适应各种图像质量和干扰情况。
transform_train = Compose([
    Resize(IMG_SIZE),
    # 随机旋转图像，旋转角度在-40到40度之间
    RandomRotation(40),
    # 随机水平翻转图像，翻转概率为0.5
    RandomHorizontalFlip(),
    ToTensor(),
    Normalize(mean=mean, std=std)
])

transform_eval = Compose([
    Resize(IMG_SIZE),
    ToTensor(),
    Normalize(mean=mean, std=std)
])

Computing mean/std: 100%|██████████| 704/704 [00:46<00:00, 15.05it/s]


In [17]:
train_ds = Cifar10Dataset(model='train', transform=transform_train)
val_ds = Cifar10Dataset(model='val', transform=transform_eval)
test_ds = Cifar10Dataset(model='test', transform=transform_eval)

batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [ ]:
model = VGG19().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

trainer = get_trainer()(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler_step="plateau",
    monitor="val_loss",
    mode="min",
    patience=5,
    device=device,
    checkpoint_dir=Path('./checkpoints/vgg19'),
)

trainer.fit(train_loader, val_loader, epochs=20)
trainer.plot_history()

train:   0%|          | 1/704 [00:49<9:41:09, 49.60s/it, loss=6.91]